In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'
import daq
import spectrum as sp
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = daq.run_log()

df_ee_ae_s_filter = lambda ee, ae, s: df.loc[(df['excitation_energy_ev']==ee)&(df['arm_energy_ev']==ae)&(df['sample']==s)]

from droplet_temperature import DropletCooling
cool = DropletCooling(diameter_um=17.9, spacing_um=153,
                      frequency_hz=120e3, pressure_pa=0.015, max_time=11e-2)

def df_y_filter(ee, ae, s, y, atol=1e-3):
    df = df_ee_ae_s_filter(ee, ae, s)
    return df.loc[(df['y'] - y).abs() < atol]

In [ ]:
import lmfit
import fitting_functions as ff


runs = df_y_filter(548.96, 525, "droplets", 4).acq.to_numpy()
run_list = [daq.load_acq(a, calibrated=True) for a in runs]
s = sp.combine(run_list).smooth_gauss(0.06)
w = s.cropped(524, 531)                               # fit window, in eV
params = ff.make_parameters([527.1, 527.45, 528.1],           # initial positions
                            amp=(0.1, 0, np.inf),         # (value, min, max), in units of the data
                            width=(0.1, 0.001, 0.5),     # Gaussian FWHM, eV
                            position_window=0.1,      # each position may move +-1 eV
                            gamma=(0.05, 0.01, 0.1))
result = lmfit.minimize(ff.residual, params, args=(w.energy, w.intensity, 2, "gauss"))
lmfit.report_fit(result)

In [ ]:
get_gauss = lambda e, res, idx: ff.gauss(e, res.params[f'position_{idx}'], res.params[f'amp_{idx}'], res.params[f'width_{idx}'])
get_voigt = lambda e, res, idx: ff.voigt(e, res.params[f'position_{idx}'], res.params[f'amp_{idx}'], res.params[f'width_{idx}'], res.params[f'gamma_{idx}'])

In [ ]:
np.trapezoid(fit, x=w.energy)

In [ ]:
plt.plot(w.energy, w.normalised().intensity)
for idx in range(len(result.params)//4):
    fit = get_gauss(w.energy, result, idx)
    fit /= np.trapezoid(fit, x=w.energy)
    plt.plot(w.energy, fit)
plt.xlabel("Energy (eV)")
plt.ylabel("Intensity (a.u.)")
plt.show()